# Loading and Running a Pre-trained LLM

In [19]:
import jax
import flax.nnx as nnx
from pathlib import Path

from helper import MiniGPT, generate_story

In [20]:
model = MiniGPT()

In [21]:
model

MiniGPT( # Param: 20,212,608 (80.9 MB)
  maxlen=128,
  embedding=TokenAndPositionEmbedding( # Param: 9,673,920 (38.7 MB)
    token_emb=Embed( # Param: 9,649,344 (38.6 MB)
      embedding=Param( # 9,649,344 (38.6 MB)
        value=Array(shape=(50257, 192), dtype=dtype('float32'))
      ),
      num_embeddings=50257,
      features=192,
      dtype=dtype('float32'),
      param_dtype=float32,
      embedding_init=<function variance_scaling.<locals>.init at 0x7f4bff0669e0>
    ),
    pos_emb=Embed( # Param: 24,576 (98.3 KB)
      embedding=Param( # 24,576 (98.3 KB)
        value=Array(shape=(128, 192), dtype=dtype('float32'))
      ),
      num_embeddings=128,
      features=192,
      dtype=dtype('float32'),
      param_dtype=float32,
      embedding_init=<function variance_scaling.<locals>.init at 0x7f4bff0669e0>
    )
  ),
  transformer_blocks=[TransformerBlock( # Param: 148,224 (592.9 KB)
    attention=MultiHeadAttention( # Param: 148,224 (592.9 KB)
      num_heads=6,
      in_feature

## Load the saved checkpoint


In [22]:
import orbax
from orbax import checkpoint

from jax.sharding import SingleDeviceSharding 

In [23]:
cpu_device = jax.devices('cpu')[0]
cpu_sharding = SingleDeviceSharding(cpu_device)

In [24]:
restore_args = jax.tree_util.tree_map(
    lambda _: checkpoint.ArrayRestoreArgs(sharding=cpu_sharding),
    nnx.state(model)
)

In [25]:
nnx.state(model)

State({
  'embedding': {
    'pos_emb': {
      'embedding': VariableState( # 24,576 (98.3 KB)
        type=Param,
        value=Array([[-0.01419522, -0.07224084, -0.03003287, ...,  0.11911442,
                 0.05112552,  0.01619032],
               [-0.01278648,  0.10531804,  0.12258674, ...,  0.02262187,
                 0.09323123, -0.00047417],
               [ 0.07250541,  0.04745221, -0.03799577, ..., -0.00529195,
                -0.09637075,  0.0727474 ],
               ...,
               [-0.04758741,  0.02744839,  0.09306912, ..., -0.07304388,
                -0.05350816, -0.03844584],
               [-0.02198958, -0.08622873,  0.10002173, ...,  0.01229017,
                 0.00219853, -0.04677548],
               [ 0.03125002,  0.03727958,  0.06220404, ..., -0.05039746,
                -0.14442575, -0.03545336]], dtype=float32)
      )
    },
    'token_emb': {
      'embedding': VariableState( # 9,649,344 (38.6 MB)
        type=Param,
        value=Array([[ 0.02099387,  0

In [ ]:
checkpoint_path = Path().resolve() / "small_checkpoint.orbax"
checkpointer = orbax.checkpoint.PyTreeCheckpointer()

In [27]:
restored_state = checkpointer.restore(
    checkpoint_path,
    item=nnx.state(model),
    restore_args=restore_args)

nnx.update(model,restored_state)

## Run inference

In [28]:
def create_story(story_prompt, temperature, max_new_tokens):
    return generate_story(model, story_prompt, temperature, max_new_tokens)

In [29]:
create_story("Once upon a time a big bear ", 0.2, 30)

'Once upon a time a big bear  All a little boy was three years old. He was very happy and wanted to play with his friends. He was very happy and he saw a big'

## Interactive chat demo

In [30]:
import gradio as gr

demo = gr.Interface(
    fn=create_story,
    inputs=[
        gr.Textbox(label="Story Prompt"),         
        gr.Slider(
            minimum=0, maximum=1.0, value=0.8, step=0.01, label="Temperature"
        ),
        gr.Slider(minimum=0, maximum=200, value=10, step=1, label="Max Tokens"
        )
    ],
    outputs=["text"]
)

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://97873d2dd7000b3dc1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
